# Initial Analysis: External Data Summary
Count unique gene pairs and unique genes in each external dataset.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, 'code')  # add code directory to path for imports

from gene_corrections_config import GENE_CORRECTIONS  # gene name corrections dict
GENE_CORRECTIONS_UPPER = {k.upper(): v.upper() for k, v in GENE_CORRECTIONS.items()}  # uppercase keys for case-insensitive lookup

def apply_gene_correction(gene):
    """Apply gene name correction, return uppercase."""
    if pd.isna(gene):
        return gene
    gene_upper = str(gene).upper()
    return GENE_CORRECTIONS_UPPER.get(gene_upper, gene_upper)  # return corrected or original

EXTERNAL_DATA_DIR = 'external data'

## Adamson Data

In [2]:
adamson = pd.read_csv(f'{EXTERNAL_DATA_DIR}/pair_sgRNA_phenotypes.txt', sep='\t')  # load Adamson sgRNA data
adamson['gene1_corrected'] = adamson['FirstGene'].str.upper().apply(apply_gene_correction)  # normalize + correct gene1
adamson['gene2_corrected'] = adamson['SecondGene'].str.upper().apply(apply_gene_correction)  # normalize + correct gene2
adamson['pair_key'] = adamson.apply(lambda r: frozenset([r['gene1_corrected'], r['gene2_corrected']]), axis=1)  # unordered pair as frozenset
adamson_genes = set(adamson['gene1_corrected'].dropna()) | set(adamson['gene2_corrected'].dropna())  # union of all genes

print(f'Adamson: {len(adamson):,} rows, {adamson["pair_key"].nunique():,} unique gene pairs, {len(adamson_genes):,} unique genes')

Adamson: 1,264,760 rows, 147,658 unique gene pairs, 543 unique genes


## Corn Data

In [3]:
corn = pd.read_csv(f'{EXTERNAL_DATA_DIR}/41586_2025_8815_MOESM5_ESM.csv')  # load Corn data
gene_splits = corn['gene_combination'].str.split(';', expand=True)  # split "GENE1;GENE2" format
corn['gene1_raw'] = gene_splits[0].str.replace('_mis', '', regex=False).str.upper()  # remove _mis suffix, uppercase
corn['gene2_raw'] = gene_splits[1].str.replace('_mis', '', regex=False).str.upper()  # same for gene2
corn = corn.dropna(subset=['gene1_raw', 'gene2_raw'])  # drop rows with missing genes
corn['gene1_corrected'] = corn['gene1_raw'].apply(apply_gene_correction)  # apply corrections
corn['gene2_corrected'] = corn['gene2_raw'].apply(apply_gene_correction)
corn['pair_key'] = corn.apply(lambda r: frozenset([r['gene1_corrected'], r['gene2_corrected']]), axis=1)  # unordered pair
corn_genes = set(corn['gene1_corrected'].dropna()) | set(corn['gene2_corrected'].dropna())  # union of all genes

print(f'Corn: {len(corn):,} rows, {corn["pair_key"].nunique():,} unique gene pairs, {len(corn_genes):,} unique genes')

Corn: 149,787 rows, 149,241 unique gene pairs, 547 unique genes


## Gilbert Data

In [4]:
gilbert_data = pd.read_excel(f'{EXTERNAL_DATA_DIR}/mmc5.xlsx',  # load Gilbert excel
                             sheet_name='gene GI scores and correlations',
                             header=[0, 1, 2],  # 3 header rows: cell line, replicate, metric
                             skiprows=[3])  # skip empty row 4

k562_col = None  # find K562 Replicate Average GI score column
for col in gilbert_data.columns:
    if col[0] == 'K562' and col[1] == 'Replicate Average' and col[2] == 'GI score':
        k562_col = col
        break

gilbert_data = pd.DataFrame({  # extract relevant columns
    'gene1': gilbert_data[gilbert_data.columns[0]],  # first col = gene1
    'gene2': gilbert_data[gilbert_data.columns[1]],  # second col = gene2
    'K562_avg': pd.to_numeric(gilbert_data[k562_col], errors='coerce')  # K562 score
})
gilbert_data['gene1'] = gilbert_data['gene1'].str.upper()  # uppercase
gilbert_data['gene2'] = gilbert_data['gene2'].str.upper()
gilbert_data = gilbert_data.dropna(subset=['gene1', 'gene2'])  # drop missing gene names
clean_gilbert = gilbert_data.dropna(subset=['K562_avg']).copy()  # keep only rows with valid K562 scores; .copy() avoids SettingWithCopyWarning

clean_gilbert['gene1_corrected'] = clean_gilbert['gene1'].apply(apply_gene_correction)  # apply corrections
clean_gilbert['gene2_corrected'] = clean_gilbert['gene2'].apply(apply_gene_correction)
clean_gilbert['pair_key'] = clean_gilbert.apply(lambda r: frozenset([r['gene1_corrected'], r['gene2_corrected']]), axis=1)
gilbert_genes = set(clean_gilbert['gene1_corrected'].dropna()) | set(clean_gilbert['gene2_corrected'].dropna())

print(f'Gilbert: {len(clean_gilbert):,} rows, {clean_gilbert["pair_key"].nunique():,} unique gene pairs, {len(gilbert_genes):,} unique genes')

Gilbert: 100,576 rows, 100,576 unique gene pairs, 448 unique genes


## Summary Table

In [5]:
summary = pd.DataFrame({
    'Dataset': ['Adamson', 'Corn', 'Gilbert'],
    'Total Rows': [len(adamson), len(corn), len(clean_gilbert)],
    'Unique Gene Pairs': [adamson['pair_key'].nunique(), corn['pair_key'].nunique(), clean_gilbert['pair_key'].nunique()],
    'Unique Genes': [len(adamson_genes), len(corn_genes), len(gilbert_genes)],
    'Max Symmetric Pairs': [n*(n-1)//2 for n in [len(adamson_genes), len(corn_genes), len(gilbert_genes)]]  # n*(n-1)/2
})
summary

,Dataset,Total Rows,Unique Gene Pairs,Unique Genes,Max Symmetric Pairs
0,Adamson,1264760,147658,543,147153
1,Corn,149787,149241,547,149331
2,Gilbert,100576,100576,448,100128
